<a href="https://colab.research.google.com/github/kancharlasaisruthi/23071A6794-DLA-ELA-1/blob/main/DLA_ELA_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Track 1: Phishing & Malicious Email Classifier

This notebook will guide you through building a Deep Learning model to classify emails as either legitimate ('ham') or potentially malicious/phishing ('spam').

### Pipeline Steps:
1.  **Data Loading**: Acquire and load a relevant email dataset.
2.  **Data Preprocessing**: Clean and prepare the email text for model input.
3.  **Model Building**: Design and implement a Deep Learning model (e.g., LSTM, GRU, or Transformer).
4.  **Model Training**: Train the model using the preprocessed data.
5.  **Evaluation**: Assess the model's performance.

# 1. Data Loading and Initial Exploration

We will use the 'SMS Spam Collection' dataset as a proxy for email classification. While it contains SMS messages, the classification task (spam vs. ham) is analogous to identifying phishing/malicious emails. We will adapt the 'spam' label to represent 'malicious/phishing' and 'ham' as 'legitimate'.

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
# For 'uciml/sms-spam-collection-dataset', the main file is 'spam.csv'
file_path = "spam.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "uciml/sms-spam-collection-dataset",
  file_path,
  pandas_kwargs={'encoding': 'latin-1'},
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipykernel_724/399778911.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
First 5 records:      v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [3]:
print(df.iloc[0]) # Accesses the second row of the DataFrame

v1                                                          ham
v2            Go until jurong point, crazy.. Available only ...
Unnamed: 2                                                  NaN
Unnamed: 3                                                  NaN
Unnamed: 4                                                  NaN
Name: 0, dtype: object


In [4]:
df[df['Unnamed: 2']==' PO Box 5249']

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
95,spam,Your free ringtone is waiting to be collected....,PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
899,spam,Your free ringtone is waiting to be collected....,PO Box 5249,"MK17 92H. 450Ppw 16""",NaN


In [5]:
df['Unnamed: 2'].unique()

array([nan, ' PO Box 5249',
       ' the person is definitely special for u..... But if the person is so special',
       ' HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE JEN XXX\\""',
       ' wanted to say hi. HI!!!\\" Stop? Send STOP to 62468"',
       'this wont even start........ Datz confidence.."', 'GN',
       '.;-):-D"',
       'just been in bedbut mite go 2 thepub l8tr if uwana mt up?loads a luv Jenxxx.\\""',
       ' bt not his girlfrnd... G o o d n i g h t . . .@"',
       ' I\'ll come up"',
       ' don\'t miss ur best life for anything... Gud nyt..."',
       ' just as a shop has to give a guarantee on what they sell. B. G."',
       ' But at d end my love compromised me for everything:-(\\".. Gud mornin:-)"',
       ' the toughest is acting Happy with all unspoken pain inside..\\""',
       ' smoke hella weed\\""', '\\" not \\"what i need to do.\\""',
       'JUST GOT PAYED2DAY & I HAVBEEN GIVEN Aå£50 PAY RISE 4MY WORK & HAVEBEEN MADE PRESCHOOLCO-ORDINATOR 2I AM FEELINGOOD

# **2. Pre Processing**

Removing column 3,4,5 and renaming the column names.

In [3]:
# Rename columns for clarity
df = df.rename(columns={'v1': 'label', 'v2': 'text'})

#  Drop unnecessary columns
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], errors='ignore')

print("DataFrame after renaming and dropping columns:")
display(df.head())
print("\nDataFrame Info after column operations:")
display(df.info())


DataFrame after renaming and dropping columns:


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



DataFrame Info after column operations:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


None

Converting categorical column to numerical column

In [4]:
# Convert labels to numerical format (0 for 'ham', 1 for 'spam')
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print("Label distribution after conversion:")
display(df['label'].value_counts())


Label distribution after conversion:


,count
label,
0,4825
1,747


Cleaning the text (removing punctuation and converting text to lower or upper case)

In [5]:
import string

# Text cleaning: lowercasing and removing punctuation
def clean_text(text):
    text = text.lower() # Lowercase the text
    text = ''.join([char for char in text if char not in string.punctuation]) # Remove punctuation
    return text

df['cleaned_text'] = df['text'].apply(clean_text)

print("DataFrame after text cleaning:")
display(df[['text', 'cleaned_text']].head())


DataFrame after text cleaning:


,text,cleaned_text
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


1.  **Stop Word Removal**: Eliminating common words (e.g., 'the', 'a', 'is') that don't carry much meaning for classification.

2.  **Tokenization**: Breaking down text into individual words or subword units.


3.  **Feature Extraction**: Converting text data into numerical features that a machine learning model can understand (e.g., TF-IDF, Word Embeddings).

Let's proceed with these steps.

In [7]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Function to remove stopwords
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

df['text_without_stopwords'] = df['cleaned_text'].apply(remove_stopwords)

print("DataFrame after removing stopwords:")
display(df[['cleaned_text', 'text_without_stopwords']].head())


DataFrame after removing stopwords:


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,cleaned_text,text_without_stopwords
0,go until jurong point crazy available only in ...,go jurong point crazy available bugis n great ...
1,ok lar joking wif u oni,ok lar joking wif u oni
2,free entry in 2 a wkly comp to win fa cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,u dun say so early hor u c already then say,u dun say early hor u c already say
4,nah i dont think he goes to usf he lives aroun...,nah dont think goes usf lives around though


###  Tokenization and Feature Extraction

We will now tokenize the text (break it into individual words) and then convert these words into numerical features using TF-IDF (Term Frequency-Inverse Document Frequency) vectorization. TF-IDF is a statistical measure that evaluates how relevant a word is to a document in a collection of documents.


1.  **Tokenization**: Mapping each unique word to an integer index.
2.  **Padding**: Ensuring all sequences have the same length by adding zeros.
3.  **Splitting Data**: Dividing the dataset into training and testing sets.

In [38]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Split data into training and testing sets
X_train_text, X_test_text, y_train, y_test = train_test_split(df['text_without_stopwords'], df['label'], test_size=0.2, random_state=42)

# Parameters for Tokenizer and Embedding Layer
max_words = 10000  # Maximum number of words to keep, based on word frequency
max_len = 100    # Maximum length of all sequences (emails)

# Initialize and fit the Tokenizer
tokenizer = Tokenizer(num_words=max_words, oov_token='<unk>') # oov_token for out-of-vocabulary words
tokenizer.fit_on_texts(X_train_text)

# Convert text to sequences of integers
X_train_sequences = tokenizer.texts_to_sequences(X_train_text)
X_test_sequences = tokenizer.texts_to_sequences(X_test_text)

# Pad sequences to ensure uniform length
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_len, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_len, padding='post', truncating='post')

print(f"Shape of X_train_padded: {X_train_padded.shape}")
print(f"Shape of X_test_padded: {X_test_padded.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train_padded: (4457, 100)
Shape of X_test_padded: (1115, 100)
Shape of y_train: (4457,)
Shape of y_test: (1115,)


In [23]:
X_train_padded

array([[   5, 1787,   32, ...,    0,    0,    0],
       [ 644, 2334, 1236, ...,    0,    0,    0],
       [  36, 1237,  524, ...,    0,    0,    0],
       ...,
       [1860, 1861,  164, ...,    0,    0,    0],
       [ 664, 1483,  964, ...,    0,    0,    0],
       [  76,   26,  190, ...,    0,    0,    0]], dtype=int32)

# 3 Build and Compile the LSTM Model

> Add blockquote



We will now define a sequential LSTM model using Keras. The model will consist of:
1.  An **Embedding layer**: To convert integer-encoded words into dense vectors.
2.  An **LSTM layer**: To process the sequential data and capture long-term dependencies.
3.  **Dense layers**: For classification.

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Define model parameters
embedding_dim = 128  # Dimension of the word embeddings

# Build the LSTM model
model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(units=128, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary classification (spam/ham)
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
print("LSTM Model Summary:")
model.summary()


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


LSTM Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# 4 Train the LSTM Model

Now we will train the compiled LSTM model using the padded training sequences and labels.

In [13]:
# Train the model
history = model.fit(
    X_train_padded,
    y_train,
    epochs=10, # Number of training epochs
    batch_size=32, # Number of samples per gradient update
    validation_split=0.1, # Use 10% of training data for validation
    verbose=1
)

print("\nModel training complete.")


Epoch 1/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 56s 342ms/step - accuracy: 0.8634 - loss: 0.4242 - val_accuracy: 0.8565 - val_loss: 0.4125
Epoch 2/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 73s 339ms/step - accuracy: 0.8671 - loss: 0.4047 - val_accuracy: 0.8565 - val_loss: 0.4124
Epoch 3/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 42s 336ms/step - accuracy: 0.8671 - loss: 0.4060 - val_accuracy: 0.8565 - val_loss: 0.4113
Epoch 4/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 83s 343ms/step - accuracy: 0.8671 - loss: 0.4010 - val_accuracy: 0.8565 - val_loss: 0.4113
Epoch 5/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 81s 338ms/step - accuracy: 0.8671 - loss: 0.4009 - val_accuracy: 0.8565 - val_loss: 0.4136
Epoch 6/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 43s 339ms/step - accuracy: 0.8671 - loss: 0.4028 - val_accuracy: 0.8565 - val_loss: 0.4126
Epoch 7/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 81s 332ms/step - accuracy: 0.8671 - loss: 0.4001 - val_accuracy: 0.8565 - val_loss: 0.4127
Epoch 8/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 42s 330ms/step - accuracy: 0.8671 - loss: 0

In [15]:
model.save('lstm.keras')

# 5. Evaluation

In [19]:
y_pred=model.predict(X_test_padded )

35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 177ms/step


In [22]:
y_pred


array([[0.14638992],
       [0.14638992],
       [0.1463899 ],
       ...,
       [0.1463899 ],
       [0.1463899 ],
       [0.1463899 ]], dtype=float32)

In [25]:
from sklearn.metrics import accuracy_score
import numpy as np

# Convert probabilities to binary predictions using a threshold (e.g., 0.5)
y_pred_binary = (y_pred >= 0.5).astype(i

                                       nt)

print(accuracy_score(y_pred_binary,y_test))

0.8654708520179372


## 6. Build and Compile the GRU Model

We will now define a sequential GRU model using Keras. The model will consist of:
1.  An **Embedding layer**: To convert integer-encoded words into dense vectors.
2.  A **GRU layer**: To process the sequential data and capture dependencies, similar to LSTM but generally faster.
3.  **Dense layers**: For classification.

In [40]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 829.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.0 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.21.0 which is incompatible.
tf-ke

In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

# Define model parameters (using the same as LSTM for consistency)
embedding_dim_gru = 128  # Dimension of the word embeddings

# Build the GRU model
model_gru = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim_gru, input_length=max_len),
    GRU(units=128, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary classification (spam/ham)
])

# Compile the model
model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
print("GRU Model Summary:")
model_gru.summary()

GRU Model Summary:


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  ):


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Train the GRU Model

Now we will train the compiled GRU model using the padded training sequences and labels.

In [42]:
# Train the GRU model
history_gru = model_gru.fit(
    X_train_padded,
    y_train,
    epochs=10, # Number of training epochs
    batch_size=32, # Number of samples per gradient update
    validation_split=0.1, # Use 10% of training data for validation
    verbose=1
)

print("\nGRU Model training complete.")

Epoch 1/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 70s 363ms/step - accuracy: 0.8641 - loss: 0.4221 - val_accuracy: 0.8565 - val_loss: 0.4249
Epoch 2/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 62s 320ms/step - accuracy: 0.8671 - loss: 0.4071 - val_accuracy: 0.8565 - val_loss: 0.4211
Epoch 3/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 40s 319ms/step - accuracy: 0.8671 - loss: 0.4010 - val_accuracy: 0.8565 - val_loss: 0.4199
Epoch 4/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 41s 318ms/step - accuracy: 0.8671 - loss: 0.4013 - val_accuracy: 0.8565 - val_loss: 0.4136
Epoch 5/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 42s 334ms/step - accuracy: 0.8671 - loss: 0.4029 - val_accuracy: 0.8565 - val_loss: 0.4114
Epoch 6/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 80s 318ms/step - accuracy: 0.8671 - loss: 0.4003 - val_accuracy: 0.8565 - val_loss: 0.4171
Epoch 7/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 40s 318ms/step - accuracy: 0.8671 - loss: 0.4055 - val_accuracy: 0.8565 - val_loss: 0.4116
Epoch 8/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 41s 324ms/step - accuracy: 0.8671 - loss: 0

In [44]:
model_gru.save('gru.keras')

### Evaluate the GRU Model

Let's evaluate the performance of the trained GRU model on the test set.

In [46]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

loss_gru, accuracy_gru = model_gru.evaluate(X_test_padded, y_test)

print(f"\nGRU Test Loss: {loss_gru:.4f}")
print(f"GRU Test Accuracy: {accuracy_gru:.4f}")



35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - accuracy: 0.8655 - loss: 0.3950

GRU Test Loss: 0.3950
GRU Test Accuracy: 0.8655


In [47]:
# Get predictions for GRU model
y_pred_gru_probs = model_gru.predict(X_test_padded)
y_pred_gru_binary = (y_pred_gru_probs >= 0.5).astype(int)
print(accuracy_score(y_test,y_pred_gru_binary))


# Confusion Matrix for GRU
print("\nGRU Confusion Matrix:")
conf_matrix_gru = confusion_matrix(y_test, y_pred_gru_binary)
print(conf_matrix_gru)

35/35 ━━━━━━━━━━━━━━━━━━━━ 5s 148ms/step
0.8654708520179372

GRU Confusion Matrix:
[[965   0]
 [150   0]]


# **------------- Accuracy of both lstm and gru are 0.8654 -------------**